# DEB - Rheinland-Pfalz

In [26]:
import zipfile
import patoolib
import os
from glob import glob
import warnings

import pandas as pd
from camelsp import get_metadata
from tqdm import tqdm
import pyproj

from camels_de1h import Station1h, get_input_path, get_nuts_id_from_provider_id

## Extract data

In [ ]:
input_path = get_input_path("DEB")

data_path1 = input_path / "rlp_q.zip"
data_path2 = input_path / "rlp_q_202201-202410.7z"

if not os.path.exists(input_path / "Q"):
    with zipfile.ZipFile(data_path1, 'r') as zip:
        os.makedirs(input_path / "Q" / "before_2022")
        zip.extractall(input_path / "Q"/ "before_2022")

    os.makedirs(input_path / "Q" / "after_2022")
    patoolib.extract_archive(str(data_path2), outdir=str(input_path / "Q" / "after_2022"), interactive=False, verbosity=-1)
    print("Data extracted.")
else:
    print("Data already extracted.")

Data already extracted.


In [3]:
# get metadata from CAMELS-DE daily
camelsp_meta = get_metadata()

## Parse data

**Timezone: UTC+1**

In [4]:
def parse_station_data(id: str, aggregate_hourly: bool) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Parse the discharge data of a station, identified by its ID.  
    Returns the metadata and the discharge data of that station.  
    If aggregate_hourly is True, the discharge data is aggregated from 15 minute 
    to hourly values.

    """
    file1 = input_path / "Q" / "before_2022" / "rlp_q" / f"{id}_Q.lila"
    file2 = input_path / "Q" / "after_2022" / f"{id}_Q.lila"

    # if a warning appears, raise it as an error
    try:
        header1 = pd.read_csv(file1, sep=";", nrows=7, header=None, encoding="latin1", usecols=[0, 1])

        data1 = pd.read_csv(file1, sep=";", skiprows=7, header=None, encoding="latin1", na_values="-", 
                            dtype={"date": str, "discharge_vol_obs": float}, usecols=[0, 1], 
                            parse_dates=["date"], names=["date", "discharge_vol_obs"], date_format="%d.%m.%Y %H:%M")
    except FileNotFoundError:
        print(f"No data in period before 2022 for station {id}.")
        header1 = None
        data1 = None

    try:
        header2 = pd.read_csv(file2, sep=";", nrows=7, header=None, encoding="latin1", usecols=[0, 1])

        data2 = pd.read_csv(file2, sep=";", skiprows=7, header=None, parse_dates=["date"], date_format="%d.%m.%Y %H:%M", 
                        usecols=[0, 1], names=["date", "discharge_vol_obs"], encoding="latin1", na_values="-")
    except FileNotFoundError:
        print(f"No data in period after 2022 for station {id}.")
        header2 = None
        data2 = None

    # check that headers are the same
    if header1 is not None and header2 is not None and not header1.equals(header2):
        print(f"Header of station {id} is different.")

    if data1 is not None and data2 is not None:
        # check that duplicated dates have the same discharge_vol_obs value
        duplicates = data1[data1.duplicated(subset="date", keep=False)]
        for date in duplicates.date:
            if not data1[data1.date == date].discharge_vol_obs.nunique() == 1:
                print(f"Station {id} has different discharge values for date {date}.")

        duplicates = data2[data2.duplicated(subset="date", keep=False)]
        for date in duplicates.date:
            if not data2[data2.date == date].discharge_vol_obs.nunique() == 1:
                print(f"Station {id} has different discharge values for date {date}.")

    # combine the data, drop duplicated dates
    data = pd.concat([data1, data2], axis=0).drop_duplicates(subset="date").sort_values("date").reset_index(drop=True)

    # convert to utc+0
    data["date"] = data.set_index("date").index.tz_localize("Etc/GMT-1").tz_convert("UTC")

    if aggregate_hourly:
        data = data.set_index("date").resample("H").mean().reset_index()

    # format the header
    if header1 is not None:
        header = header1
    else:
        header = header2
    header = header.T
    header.columns = header.iloc[0]
    header = header.drop(0, axis=0)

    return header, data

Use the function to parse data for all stations and save raw metadata from the header.  
Use raw metadata from CAMELS-DE daily to get metadata for stations that did not make it into the final CAMELS daily dataset.

In [ ]:
ids1 = sorted([f.split("/")[-1].split("_")[0] for f in glob(str(input_path / "Q" / "before_2022" / "rlp_q" / "*.lila"))])
ids2 = sorted([f.split("/")[-1].split("_")[0] for f in glob(str(input_path / "Q" / "after_2022" / "*.lila"))])

# combine the ids
ids = sorted(list(set(ids1 + ids2)))

# get the metadata from CAMELS-DE daily
camelsp_meta_all = get_metadata()

# read the old metadata from the excel file
meta_old_raw_all = pd.read_excel(str(input_path / "aktive Pegel_Februar 2022_mit Stammdaten.xlsx"), header=2)
meta_old_raw_all.rename(columns={'km': 'km oh. Münd.'}, inplace=True)
metameta_old_raw_all = meta_old_raw_all.iloc[1:]
# drop columns where Gewässer is NaN -> all station have a Gewässer but there are some rows which do not contain stations -> drop them by dropping rows with NaN in Gewässer
meta_old_raw_all = meta_old_raw_all.dropna(subset=['Gewässer'])
# according to rlp websites, all Messstellennummern are missing two zeros at the end
meta_old_raw_all['Nummer'] = (meta_old_raw_all['Nummer'] * 100).astype(str)

for id in tqdm(ids):
    # get the gauge_id from the nuts_mapping, add it if it does not exist
    gauge_id = get_nuts_id_from_provider_id(f"{id}00", "DEB", add_missing=True)

    # parse station metadata and data
    header, data = parse_station_data(id, aggregate_hourly=True)

    # get the metadata of the station
    camelsp_meta = camelsp_meta_all[camelsp_meta_all["camels_id"] == gauge_id]
    meta_old_raw = meta_old_raw_all[meta_old_raw_all["Nummer"] == f"{id}00"]

    if not camelsp_meta.empty:
        # set metadata values from camelsp metadata
        station_meta = {
            "gauge_id": camelsp_meta["camels_id"].values[0],
            "provider_id": camelsp_meta["provider_id"].values[0],
            "gauge_name": camelsp_meta["gauge_name"].values[0],
            "waterbody_name": camelsp_meta["waterbody_name"].values[0],
            "federal_state": camelsp_meta["federal_state"].values[0],
            "gauge_lon": camelsp_meta["lon"].values[0],
            "gauge_lat": camelsp_meta["lat"].values[0],
            "gauge_easting": camelsp_meta["x"].values[0],
            "gauge_northing": camelsp_meta["y"].values[0],
            "gauge_elev_metadata": camelsp_meta["gauge_elevation"].values[0],
            "area_metadata": camelsp_meta["area"].values[0],
            "part_of_camelsp": True,
        }
    elif not meta_old_raw.empty:
        # parse the location
        easting, northing = meta_old_raw["RW"].values[0], meta_old_raw["HW"].values[0]

        transformer = pyproj.Transformer.from_crs("epsg:31466", "epsg:4326", always_xy=True)
        lon, lat = transformer.transform(easting, northing)

        # from EPSG:31466 to EPSG:3035
        transformer = pyproj.Transformer.from_crs("epsg:31466", "epsg:3035", always_xy=True)
        easting, northing = transformer.transform(easting, northing)

        # if the station is not in the camelsp metadata, use the raw metadata of camelsp / CAMELS-DE
        station_meta = {
            "gauge_id": gauge_id,
            "provider_id": f"{id}00",
            "gauge_name": meta_old_raw["Stationsname"].values[0],
            "waterbody_name": meta_old_raw["Gewässer"].values[0],
            "federal_state": "Rheinland-Pfalz",
            "gauge_lon": lon,
            "gauge_lat": lat,
            "gauge_easting": easting,
            "gauge_northing": northing,
            "gauge_elev_metadata": meta_old_raw["PNP"].values[0],
            "area_metadata": meta_old_raw["Aeo"].values[0],
            "part_of_camelsp": False,
        }
    else:
        # if the station is also not in the excel metadata from CAMELS-DE, use the raw metadata from the .lila files
        station_meta = {
            "gauge_id": gauge_id,
            "provider_id": f"{id}00",
            "gauge_name": header["Station"].values[0],
            "waterbody_name": None,
            "federal_state": "Rheinland-Pfalz",
            "gauge_lon": None,
            "gauge_lat": None,
            "gauge_easting": None,
            "gauge_northing": None,
            "gauge_elev_metadata": None,
            "area_metadata": None,
            "part_of_camelsp": False,
        }
        
    # initialize the station
    station = Station1h(gauge_id)

    # save time series data
    station.save_data(data)

    # save metadata
    station.save_metadata(**station_meta)
    
    # save raw metadata
    station.save_raw_metadata(header)

  1%|          | 1/134 [00:03<07:11,  3.24s/it]

Station 23730116 not in CAMELS-DE daily.


  6%|▌         | 8/134 [00:29<09:15,  4.41s/it]

Station 23780602 not in CAMELS-DE daily.


  7%|▋         | 10/134 [00:35<07:44,  3.74s/it]

...

 92%|█████████▏| 123/134 [07:34<00:36,  3.34s/it]

Station 27170909 not in CAMELS-DE daily.


 94%|█████████▍| 126/134 [07:42<00:23,  2.92s/it]

Station 27180607 not in CAMELS-DE daily.


100%|██████████| 134/134 [08:19<00:00,  3.73s/it]
